In [2]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import create_urllib3_context
import json
import csv

In [27]:

# 1. 建立一個自訂的 SSL 上下文管理器，允許不安全的重新協商
class LegacyVersionAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        context = create_urllib3_context()
        # 💡 核心：將 SSL 的 Options 加上允許不安全重新協商的旗標
        # 0x40000 代表 SSL_OP_LEGACY_SERVER_CONNECT
        context.options |= 0x40000  
        kwargs["ssl_context"] = context
        return super().init_poolmanager(*args, **kwargs)

def fetch_nccu_courses(url): 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Origin": "https://qrysub.nccu.edu.tw",
        "Referer": "https://qrysub.nccu.edu.tw/"
    }
    
    session = requests.Session()
    session.mount("https://", LegacyVersionAdapter())
    
    try:
        response = session.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error: {e}")
        return None

    

parameters = ["01", "02", "03", "100", "200", "300", "400", "500", "600", "700", "800", "900", "Z23", "ZA0", "ZC0"]
data = []
course_id = set()
for parameter in parameters: 
    response = fetch_nccu_courses(f"https://es.nccu.edu.tw/course/zh-TW/:sem=1141%20:dp1={parameter}%20/")
    for c in response:
        if c["subNum"] not in course_id:
            data.append(c)
            course_id.add(c["subNum"])
            
    response = fetch_nccu_courses(f"https://es.nccu.edu.tw/course/zh-TW/:sem=1142%20:dp1={parameter}%20/")
    for c in response:
        if c["subNum"] not in course_id:
            data.append(c)
            course_id.add(c["subNum"])

print(len(data))

5339


In [20]:
with open("response.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [21]:
gdeType = set()
lmtKind = set()
tranTpe = set()
subGde  = set()
subKind = set()

for course in data:
    if ("學士" in course["gdeType"]):
        gdeType.add(course["gdeType"])
        lmtKind.add(course["lmtKind"])
        subGde.add(course["subGde"])
        subKind.add(course["subKind"])

print(f"gdeType: {gdeType}")
print(f"lmtKind: {lmtKind}")
print(f"subGde: {subGde}")
print(f"subKind: {subKind}")

gdeType: {'學士班', '學士班、碩士班'}
lmtKind: {'', '自然通識', '跨領域(社會、自然、資訊)', '跨領域(人文、社會、自然)', '中文通識', '跨領域(人文、社會)', '跨領域(人文、資訊)', '跨領域(社會、資訊)', '人文通識', '跨領域(自然、資訊)', '跨領域(社會、自然)', '社會通識', '外文通識', '跨領域(人文、自然)', '資訊通識', '書院通識'}
subGde: {'財管四', '中文四甲', '歐文法三', '土文三土文四', '地二土管地二土資', '日文三', '統計一', '企管一乙', '法學院', '外交系', '心理三心理四', '地二土管地二土資地二土測', '風管二風管三風管四', '歐文法三歐文法四', '歷史三', '風管三風管四', '歐文法三歐文法四歐文法選', '土文四', '斯語一甲', '民族四', '台史所', '風管三', '地二土測', '風管四', '資管系', '日文系', '東南越二', '華語文中心', '國貿二甲國貿二乙國貿三甲國貿三乙國貿四甲國貿四乙', '教育三', '會計三甲', '英文一甲', '歐文德四', '中文二乙', '公行三公行四', '財政一乙', '斯語三甲斯語三乙斯語四甲斯語四乙', '斯語一甲斯語一乙', '金融三', '東南越二東南泰二', '經濟二甲經濟二乙經濟三甲經濟三乙經濟四甲經濟四乙', '韓文三韓文四', '教育四', '中文二甲中文二乙', '教院學士', '歐文西三', '國貿三甲國貿三乙國貿四甲國貿四乙', '廣告系', '財政四甲財政四乙', '資訊二資訊三資訊四', '民族三', '歐文德三', '東南泰三東南印三東南泰四東南印四', '政治系', '歐文法二歐文德二歐文西二歐文法三歐文德三歐文西三歐文法四歐文德四歐文西四歐文法選歐文德選歐文西選', '斯語二甲', '體育', '斯語二乙', '會計二乙', '土文一', '公行一公行二公行三公行四', '心理四', '地三土管地三土資', '教育系', '會計三乙', '企管二甲企管二乙', '歷史一', '傳播院學', '文院學士', '財管三財管四', '傳院二甲傳院二乙傳院二丙傳院二丁', '經濟三甲經濟三乙經濟四甲經濟四乙', '

In [28]:
department_keywords = {
    '文學院': '100', '中文': '101', '教育': '102', '歷史': '103', '哲學': '104',
    '政治': '202', '外交': '203', '社會': '204', '財政': '205', '公行': '206', '地': '207', '經濟': '208', '民族': '209', 
    '國貿': '301', '金融': '302', '會計': '303', '統計': '304', '企管': '305', '資管': '306', '財管': '307', '風管': '308',
    '新聞': '401', '廣告': '402', '廣電': '403', '傳院': '405', '傳播': '405',
    '英文': '501', '阿文': '502', '斯語': '504', '日文': '506', '韓文': '507', '土文': '508', '歐文': '509', '東南泰': '510',
    '法律': '601',
    '應數': '701', '心理': '702', '資訊': '703',
    '創國': 'ZU1',
    '體育': 'B00', '華語文': "162", "外文中": "5T1"
}

course_type_keywords = {
    '群修': 'P', '必修': 'R', '選修': 'E', 
    "中文通識": "GC", "外文通識": "GF", 
    "人文通識": "GH", "社會通識": "GS", "自然通識": "GN", "資訊通識": "GI", "書院通識": "GA",
    "跨領域(社會、自然)": "GSN", "跨領域(人文、社會)": "GHS", "跨領域(人文、自然)": "GHN", "跨領域(人文、資訊)": "GHI", "跨領域(社會、資訊)": "GSI", "跨領域(自然、資訊)": "GNI",
    "跨領域(社會、自然、資訊)": "GSNI", "跨領域(人文、社會、自然)": "GHSN",
}

course_records = [['course_id', 'course_name', 'course_type', 'teacher_name', 'department_id', 'credits']]
for course in data:
    if ("學士" in course["gdeType"]):
        course_id = course["subNum"]
        course_name = course["subNam"]
        if "體育" in course_name:
            course_type = course_type_keywords[course["subKind"]] + "PE"        
        elif "國防" in course_name:
            course_type = "CD"  
        elif course["subKind"]:
            course_type = course_type_keywords[course["subKind"]]
        else:
            course_type = course_type_keywords[course["lmtKind"]]
        if course["core"] == "是":
            course_type = "C" + course_type
            
        teacher_name = course["teaNam"]
        
        for key, val in department_keywords.items():
            if key in course["subGde"]:
                department_id = str(val)
                break
        else:
            department_id = "000"
        credits = int(float(course["subPoint"]))      
        
        course_records.append([
            course_id,
            course_name,
            course_type,
            teacher_name,
            department_id,
            credits 
        ])

with open('course_information.csv', 'w', encoding='utf-8', newline="") as f:
    writer = csv.writer(f)
    writer.writerows(course_records)